# Molecular Docking from Scratch — with Open-Source Code
### A hands-on tutorial (no GUI, every step visible)

You've already docked FA19 in MOE. This notebook teaches you to do it yourself in **fully open-source code**, so you understand the *computation*, not just the button. By the end you will have:
1. docked a real drug into a real protein (a 2-minute warm-up), and
2. built a COX-2 + FA19 docking **from scratch** — prepare, dock, **validate**, and visualise.

---

## The one idea behind all docking

**Docking = SEARCH + SCORE.**

- **Search:** try many *poses* of the ligand — its position, orientation, and the twists of its rotatable bonds — inside a defined region of the protein.
- **Score:** for each pose, estimate the binding strength with a *scoring function* (kcal/mol; more negative = predicted tighter binding).

The docking engine (AutoDock Vina) repeats search→score thousands of times and returns the best-scoring poses. Everything else in this notebook is just *preparing the inputs* so the engine can do that well.

> Scoring functions are **approximations**. Treat the number as a way to *rank and prioritise*, not as a true free energy. The trustworthiness of your run is established by **re-docking a known ligand** (Section 2.5) — that is the part most beginners skip and shouldn't.


## The open-source toolchain (what replaces the GUI)

| Job | GUI does it invisibly | Open-source tool you'll use |
|---|---|---|
| Get the protein structure | bundled | download `.pdb` from the RCSB |
| Clean + protonate receptor | a checkbox | Open Babel / PDBFixer |
| Convert to the docking format (PDBQT) | hidden | Open Babel / Meeko |
| Build a 3D ligand from a drawing | a sketcher | RDKit (from SMILES) |
| Prepare ligand (PDBQT) | hidden | Meeko |
| Define the search box | drag a box | you set center + size explicitly |
| Run the docking | "Run" | **AutoDock Vina** |
| Look at the result | 3D viewer | py3Dmol (in this notebook) |

**PDBQT** is the key format: it's a PDB file **plus Partial charges (Q) plus AutoDock atom Types (T)**, and for ligands it also encodes which bonds are rotatable. That's literally everything Vina needs to score a pose.


## Setup

In [ ]:
!pip -q install vina meeko rdkit py3Dmol numpy requests 2>/dev/null
!apt-get -qq install -y openbabel >/dev/null 2>&1
print("ready")

## Section 1 — See it work first (2-minute warm-up)

Before the concepts, let's just *do* a docking so you see the payoff. We'll use the official Vina benchmark: **imatinib (Gleevec) into Abl kinase, PDB 1IEP** — the receptor and ligand are already prepared as PDBQT, so this is pure search+score.


In [ ]:
# Grab the official Vina example (already-prepared PDBQT files)
!git clone -q --depth 1 https://github.com/ccsb-scripps/AutoDock-Vina.git /content/advina
base = "/content/advina/example/basic_docking/solution"

from vina import Vina
v = Vina(sf_name='vina', cpu=2, seed=42, verbosity=1)
v.set_receptor(f"{base}/1iep_receptor.pdbqt")          # the protein (rigid)
v.set_ligand_from_file(f"{base}/1iep_ligand.pdbqt")     # the drug (flexible)

# the search box for this site (center in Angstrom, cubic 20 A box)
v.compute_vina_maps(center=[15.19, 53.90, 16.92], box_size=[20, 20, 20])
v.dock(exhaustiveness=8, n_poses=9)                     # SEARCH + SCORE
energies = v.energies(n_poses=9)
print("\nPose   affinity (kcal/mol)")
for i, row in enumerate(energies, 1):
    print(f"  {i:>2}        {row[0]:7.2f}")
print("\nBest pose:", round(float(energies[0][0]), 2), "kcal/mol")
# Expected: about -13 kcal/mol (imatinib is a tight binder). You just docked a drug.

**What just happened:** Vina took a rigid protein and a flexible ligand, searched the 20 Å box, scored thousands of poses, and ranked the best 9. The top pose (~ −13 kcal/mol) is very favourable — consistent with imatinib being a potent Abl inhibitor.

Now we build the same thing for **your** system, doing the preparation ourselves so you learn each step.


## Section 2 — Build a COX-2 + FA19 docking from scratch

### 2.1 The receptor

A crystal structure (PDB **3LN1** = COX-2 with celecoxib bound) contains things Vina must *not* see: water, the bound drug, ions, sometimes multiple protein copies. We **clean** it (keep one protein chain, drop everything else), **protonate** it (add hydrogens — protonation states drive H-bonding), and convert to **PDBQT** (adds charges + atom types). A receptor is treated as **rigid** by default.


In [ ]:
import requests
pdb = requests.get("https://files.rcsb.org/download/3LN1.pdb", timeout=60).text
open("3LN1.pdb","w").write(pdb)

# KEEP chain A protein atoms only (ATOM records); DROP waters/ligands/ions (HETATM)
with open("receptor_clean.pdb","w") as f:
    for line in pdb.splitlines():
        if line.startswith("ATOM") and line[21] == "A":
            f.write(line+"\n")
    f.write("TER\nEND\n")

# -> PDBQT: -xr = rigid receptor, -p 7.4 = add H at physiological pH (assigns charges+types)
!obabel receptor_clean.pdb -O receptor.pdbqt -xr -p 7.4 2>/dev/null
import os; print("receptor.pdbqt:", os.path.getsize("receptor.pdbqt"), "bytes  (ready)")

### 2.2 The ligand

Your GUI lets you sketch a molecule; here we build it from a **SMILES string** — a text encoding of the structure. RDKit turns SMILES into a sensible **3D** conformer (embed + energy-minimise), then Meeko writes the **PDBQT**, automatically detecting **rotatable bonds** (the torsions Vina will search).


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

FA19 = "COc1ccc(C(=O)Oc2ccc(/C=C/C(=O)O)cc2OC)cc1"   # your lead, as SMILES
m = Chem.AddHs(Chem.MolFromSmiles(FA19))             # add explicit hydrogens
AllChem.EmbedMolecule(m, randomSeed=42)              # generate a 3D conformer
AllChem.MMFFOptimizeMolecule(m)                      # relax it with a force field

from meeko import MoleculePreparation, PDBQTWriterLegacy
setups = MoleculePreparation().prepare(m)            # assign charges/types/torsions
pdbqt_str = PDBQTWriterLegacy.write_string(setups[0])[0]
open("fa19.pdbqt","w").write(pdbqt_str)
# count rotatable bonds Vina will explore (the BRANCH lines in the torsion tree)
print("rotatable bonds (torsions):", pdbqt_str.count("BRANCH")//2)
print("fa19.pdbqt ready")

### 2.3 The search box — the most important choice you make

Vina only searches *inside the box you define*. Put it in the wrong place and you get a confident, precise, **wrong** answer. The honest, reproducible way to place it: **center the box on a ligand already known to bind there** — here, the celecoxib co-crystallised in 3LN1. Size it to comfortably fit your ligand plus room to move (~22–25 Å is typical for a drug-sized molecule).


In [ ]:
import numpy as np
# pull the bound celecoxib (resname CEL, chain A) straight from the crystal file
xyz = [[float(l[30:38]), float(l[38:46]), float(l[46:54])]
       for l in pdb.splitlines()
       if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A"]
center = np.mean(xyz, axis=0).round(2).tolist()
print("Box center (centroid of the known binder):", center)
BOX = [24.0, 24.0, 24.0]
print("Box size:", BOX, "Angstrom")

### 2.4 Dock

A few knobs worth understanding:
- **scoring function** (`vina`): empirical — sums steric, hydrophobic, H-bond, and a torsion penalty. Output in kcal/mol.
- **exhaustiveness**: how hard Vina searches. Higher = better chance of finding the true best pose, but slower. 8 is the default; 16–32 for a careful run.
- **seed**: the search is stochastic, so **fix a seed** to make runs reproducible (and always report it).


In [ ]:
v = Vina(sf_name='vina', cpu=2, seed=42, verbosity=1)
v.set_receptor("receptor.pdbqt")
v.set_ligand_from_file("fa19.pdbqt")
v.compute_vina_maps(center=center, box_size=BOX)
v.dock(exhaustiveness=16, n_poses=10)
v.write_poses("fa19_out.pdbqt", n_poses=5, overwrite=True)
fa19_best = float(v.energies(n_poses=1)[0][0])
print("\nFA19 best Vina affinity:", round(fa19_best,2), "kcal/mol")
print("Your MOE value was -8.3 kcal/mol. Different scoring functions won't match exactly;")
print("agreement within ~1 kcal/mol is the meaningful check (this is cross-validation, not a contradiction).")

### 2.5 Validate the setup — the step beginners skip

A good FA19 number is meaningless if the *protocol* is wrong. The test: **re-dock the known ligand (celecoxib) and see if you recover its crystal pose.** If the redocked pose lands within **~2 Å RMSD** of the real crystal position, your box, receptor prep, and engine are trustworthy. If it's 5+ Å, fix the setup before believing anything.


In [ ]:
# extract crystal celecoxib, prepare it, dock it back into the same box
with open("cel_crystal.pdb","w") as f:
    for l in pdb.splitlines():
        if l.startswith("HETATM") and l[17:20].strip()=="CEL" and l[21]=="A":
            f.write(l+"\n")
    f.write("END\n")
!obabel cel_crystal.pdb -O cel.pdbqt -p 7.4 2>/dev/null
!obabel cel_crystal.pdb -O cel_crystal.sdf 2>/dev/null

vc = Vina(sf_name='vina', cpu=2, seed=42, verbosity=0)
vc.set_receptor("receptor.pdbqt"); vc.set_ligand_from_file("cel.pdbqt")
vc.compute_vina_maps(center=center, box_size=BOX)
vc.dock(exhaustiveness=16, n_poses=5)
vc.write_poses("cel_redock.pdbqt", n_poses=1, overwrite=True)
!obabel cel_redock.pdbqt -O cel_redock.sdf -f 1 -l 1 2>/dev/null

print("Redocked celecoxib affinity:", round(float(vc.energies(n_poses=1)[0][0]),2), "kcal/mol")
print("\nPose RMSD vs the crystal (want < 2.0 A):")
!obrms cel_crystal.sdf cel_redock.sdf

### 2.6 See the result in 3D

Numbers are abstract; *look* at the pose. py3Dmol renders the protein with your docked FA19 inside the pocket — rotate it, and you can see whether it sits where celecoxib did.


In [ ]:
import py3Dmol
view = py3Dmol.view(width=750, height=520)
view.addModel(open("receptor_clean.pdb").read(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
# docked FA19 (first pose) as sticks
fa19_pose = open("fa19_out.pdbqt").read()
view.addModel(fa19_pose, "pdbqt")
view.setStyle({"model": -1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": -1})
view.show()

## Section 3 — Now experiment (this is how you learn)

Change one thing, rerun, observe. Suggested experiments:

1. **Box size.** Shrink `BOX` to `[12,12,12]` or grow to `[34,34,34]`. Too small clips the ligand; too large wastes the search and can dock into the wrong sub-site. Watch the affinity and pose change.
2. **Exhaustiveness.** Drop to `2`, then push to `32`. Low values give noisier, less reproducible results; note the runtime trade-off.
3. **Seed.** Run twice with the same seed (identical) vs different seeds (slightly different) — this is what "stochastic search" means.
4. **A different ligand.** Swap the `FA19` SMILES for another derivative from your library and compare. This is exactly how you'd dock all 24 and rank them.
5. **Re-dock control first, always.** Make Section 2.5 a habit: if the RMSD is bad, no other number is trustworthy.

---

## How this fits with your existing work

- **You keep your MOE results.** This open-source run is a *cross-validation* and a learning exercise — two independent methods agreeing is stronger evidence than one.
- This is the same engine as your `FA19_COX2_Vina_redocking.ipynb`, but here you've built and understood every step by hand.
- To go deeper: read about the Vina scoring function (Trott & Olson, 2010), try **smina** or **gnina** (a CNN-rescoring fork), and learn **PyMOL** (open-source) for publication-quality pose figures.

You now understand docking as *search + score over a defined box, validated by re-docking* — the core of structure-based drug design.
